## Foldseek API Test

In [1]:
import requests
import time

def final_foldseek_test():

    print("1. Fetching 1UBQ (Ubiquitin) from RCSB in memory...")
    pdb_response = requests.get("https://files.rcsb.org/download/1UBQ.pdb")
    if pdb_response.status_code != 200:
        print("Failed to download PDB from RCSB.")
        return
    pdb_data = pdb_response.text

    print("2. Submitting structure to Foldseek...")
    submit_url = "https://search.foldseek.com/api/ticket"

    files = {
        'q': ('1UBQ.pdb', pdb_data)
    }

    data = [
        ('mode', '3diaa'),
        ('database[]', 'pdb100'),
        ('database[]', 'afdb50')
    ]

    response = requests.post(submit_url, files=files, data=data)

    if response.status_code != 200:
        print(f"Error submitting job: {response.status_code} - {response.text}")
        return

    ticket_id = response.json().get('id')
    print(f"Success! Ticket ID: {ticket_id}")

    print("3. Waiting for server to process...")
    status_url = f"https://search.foldseek.com/api/ticket/{ticket_id}"

    while True:
        status_res = requests.get(status_url).json()
        status = status_res.get('status')

        if status == 'COMPLETE':
            print("Job complete!\n")
            break
        elif status in ['ERROR', 'FAILED']:
            print("Job failed on the server side.")
            return

        time.sleep(2)

    result_url = f"https://search.foldseek.com/api/result/{ticket_id}/0"
    results_res = requests.get(result_url)

    if results_res.status_code != 200:
        print(f"Error fetching results: {results_res.status_code} - {results_res.text}")
        return

    results = results_res.json()

    if isinstance(results, dict) and 'results' in results:
        results = results['results']

    if isinstance(results, dict) and 'db' not in results:
        print("Server returned an unexpected response/error:")
        print(results)
        return

    if isinstance(results, dict):
        results = [results]

    for db_result in results:
        if not isinstance(db_result, dict):
            continue

        db_name = db_result.get('db', 'Unknown DB')
        alignments = db_result.get('alignments', [])

        print(f"--- Results for Database: {db_name.upper()} ---")
        if not alignments:
            print("  No structural matches found.\n")
        else:
            top_hit = alignments[0]
            print(f"  Total Matches Found: {len(alignments)}")
            print(f"  Top Match Target   : {top_hit[0]}")
            print(f"  Sequence Identity  : {top_hit[6]}")
            print(f"  E-value            : {top_hit[5]}")
            print(f"  Aligned Length     : {top_hit[7]}\n")

final_foldseek_test()

1. Fetching 1UBQ (Ubiquitin) from RCSB in memory...
2. Submitting structure to Foldseek...
Success! Ticket ID: T4NsEYL4cMWIPGrMxmJM7_K5kChxG9nWjyqjAg
3. Waiting for server to process...
Job complete!

--- Results for Database: AFDB50 ---
  Total Matches Found: 1
  Top Match Target   : {'query': 'job_A', 'target': 'AF-A0A8D0HA87-F1-model_v6 Ubiquitin-like domain-containing protein', 'seqId': 100, 'alnLength': 76, 'missmatches': 0, 'gapsopened': 0, 'qStartPos': 1, 'qEndPos': 76, 'dbStartPos': 1, 'dbEndPos': 76, 'prob': 1, 'eval': 1.28e-13, 'score': 546, 'qLen': 76, 'dbLen': 77, 'qAln': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG', 'dbAln': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG', 'tCa': '0.163,5.079,10.676,-1.610,1.727,10.442,-3.315,0.473,7.227,-4.880,-2.853,6.178,-4.129,-5.000,3.100,-6.768,-7.579,2.111,-5.554,-10.436,-0.115,-7.580,-12.437,-2.707,-7.807,-15.331,-0.152,-9.578,-12.949,2.298,-6.504,-12.738,4.637,-6.079

## Swiss-Model API Test

In [2]:
import requests

def run_swiss_model(api_token):
    print("--- SWISS-MODEL API Submission ---")
    url = "https://swissmodel.expasy.org/automodel"

    headers = {
        "Authorization": f"Token {api_token}",
        "Content-Type": "application/json"
    }

    payload = {
        "target_sequences": ["MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"],
        "project_title": "API_Connection_Test"
    }

    print("Submitting sequence...")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code in [200, 201, 202]:
        try:
            project_id = response.json().get('project_id')
            print(f"\nSuccess! Job accepted by the server.")
            print(f"Project ID: {project_id}")
            print(f"Track it live here: https://swissmodel.expasy.org/project/{project_id}")
        except Exception as e:
            print("Job accepted, but failed to parse the project ID.")
            print(response.text)

    elif response.status_code == 401:
        print("\nFailed: 401 Unauthorized.")
        print("Double-check that you copied the API token perfectly with no extra spaces.")
    elif response.status_code == 429:
        print("\nFailed: 429 Rate limit exceeded. Try again in a few minutes.")
    else:
        print(f"\nFailed to submit. HTTP Status code: {response.status_code}")
        print(f"Raw Server Response: {response.text}")

YOUR_TOKEN = "31f671a6084d2d6ec8b4959ea9e274f0b8a7c88b"

run_swiss_model(YOUR_TOKEN)

--- SWISS-MODEL API Submission ---
Submitting sequence...

Success! Job accepted by the server.
Project ID: ad8b47
Track it live here: https://swissmodel.expasy.org/project/ad8b47
